# 01 · Données FHIR : téléchargement, préfiltrage, échantillon

On part des 555 dossiers patients synthétiques de Synthea (FHIR R4). Ce notebook :

1. télécharge et extrait l'archive dans `data/raw/` ;
2. préfiltre chaque dossier pour ne garder que le patient, ses conditions et ses prescriptions ;
3. tire l'échantillon de patients sur lequel portera tout le benchmark.

Pas besoin de GPU ici : tout est déterministe. Le code est dans `src/fhir.py`, les paramètres dans `config/settings.py`.

In [ ]:
# Sur Google Colab seulement : récupérer le dépôt et installer les dépendances
# !git clone https://github.com/brettsamy-database/medical-data-reconstruction.git bupa
# %cd bupa/notebooks
# !pip -q install -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

# Pour pouvoir importer `config` et `src`, que le notebook soit lancé depuis notebooks/ ou depuis la racine
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import JSON, display

from config import settings

settings.ensure_dirs()
pd.set_option("display.max_colwidth", 200)
print("ROOT:", ROOT)

## Paramètres

Les valeurs par défaut sont dans `config/settings.py` et peuvent être modifiées ici avant de lancer les étapes.
Le benchmark publié porte sur 40 patients (`N_PATIENTS = 40`) ; 5 suffisent pour un essai rapide.

In [ ]:
# settings.N_PATIENTS = 5   # essai rapide
print("N_PATIENTS:", settings.N_PATIENTS, "| RANDOM_SEED:", settings.RANDOM_SEED)
print("PREFILTER_DROP_SUBSTRINGS:", settings.PREFILTER_DROP_SUBSTRINGS)

## 1. Télécharger Synthea

In [ ]:
from src import fhir

# fhir.reset_filtered_dataset()  # après avoir modifié les règles PREFILTER_*, pour refaire le filtrage
fhir.download_synthea()

## 2. Préfiltrer les dossiers

Le préfiltre est volontairement prudent : il garde les conditions codées en SNOMED ou ICD-10, retire les
déterminants sociaux évidents (emploi, logement, violence…), supprime les doublons et plafonne le nombre de
candidats. Le tri fin vient plus tard, avec le LLM. Le résultat est écrit une fois pour toutes sur disque.

In [ ]:
df_index = fhir.build_filtered_dataset()
print("Filtered patients:", len(df_index))
display(df_index.head(10))
display(df_index[["n_conditions", "n_medication_requests"]].describe())

## 3. Tirer l'échantillon

Le tirage est reproductible grâce à `RANDOM_SEED` : le notebook 02 retrouvera exactement les mêmes patients.

In [ ]:
df_sample, patients_prefilter, df_gold_prefilter = fhir.select_sample(df_index)
print("Prefiltered sample (candidates before LLM filtering)")
display(df_gold_prefilter)

In [ ]:
# Un dossier après préfiltrage
example_pid = df_gold_prefilter.iloc[0]["patient_id"]
display(JSON(fhir.load_bundle(str(settings.FILTERED_DIR / f"{example_pid}.json"))))

**Suite :** `02_note_generation.ipynb`, qui construit le gold source, génère les notes et les reconstruit avec MedGemma.